In [1]:
# Run this cell to install the required packages
!pip install google-adk requests geopy
# Install the Google Cloud AI Platform SDK with Agent Engine and ADK support
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.112"

Bash command to create a cloud storage bucket


In [2]:
# Create a staging bucket in your project's region
!gcloud storage buckets create gs://qwiklabs-gcp-04-23b9cb055c5f-agent-staging --location=us-central1

Creating gs://qwiklabs-gcp-04-23b9cb055c5f-agent-staging/...
ERROR: (gcloud.storage.buckets.create) HTTPError 409: Your previous request to create the named bucket succeeded and you already own it.


Evironmental Variables

In [3]:
import os

os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-04-23b9cb055c5f"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"


Tool to find location and check the weather for that location from NWS

In [4]:
import requests
from geopy.geocoders import Nominatim

def get_nws_weather(city: str) -> str:
    """
    Retrieves the 7-day weather forecast for a specified US city using the National Weather Service API.

    Args:
        city (str): The name of the US city (e.g., "Seattle, WA", "New York, NY").

    Returns:
        str: A detailed description of the weather forecast for the upcoming week.
    """
    try:
        # 1. Convert the city name into coordinates
        geolocator = Nominatim(user_agent="adk_weather_lab")
        location = geolocator.geocode(city)

        if not location:
            return f"Error: Could not find coordinates for {city}."

        lat, lon = round(location.latitude, 4), round(location.longitude, 4)

        # 2. Call NWS to get the grid forecast URL
        headers = {"User-Agent": "(adk-student-lab, student@example.com)"}
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers).json()

        if "properties" not in points_resp:
            return "Error: Location may be outside the US. NWS only covers US territories."

        forecast_url = points_resp["properties"]["forecast"]

        # 3. Call the forecast URL and extract the 7-day period array
        forecast_resp = requests.get(forecast_url, headers=headers).json()
        periods = forecast_resp["properties"]["periods"]

        # Compile the full week's forecast (up to 14 day/night periods) to pass back to the LLM
        forecast_summary = [f"Forecast for {city}:"]
        for period in periods:
            forecast_summary.append(f"- {period['name']}: {period['detailedForecast']}")

        return "\n".join(forecast_summary)

    except Exception as e:
        return f"Error fetching weather: {str(e)}"

Agent Setup

In [5]:
from google.adk.agents import Agent

weather_agent = Agent(
    name="weather_bot",
    model="gemini-2.5-flash",
    description="An agent that provides accurate weather forecasts.",
    instruction="""
    You are a helpful meteorology assistant.
    Always use the get_nws_weather tool when a user asks about the weather.

    CRITICAL FORMATTING RULES:
    1. The weather tool returns a full 7-day forecast.
    2. DEFAULT RULE: If the user DOES NOT specify a day or time (e.g., "What is the weather in Seattle?"), you MUST ONLY provide the forecast for the current day and night.
    3. SPECIFIC DAY RULE: If the user specifies a day (e.g., "What is the weather on Wednesday?"), extract and provide ONLY the forecast for that specific day.
    4. Never dump the entire 7-day forecast unless the user explicitly asks for "the week".
    """,
    tools=[get_nws_weather]
)

print(f"Agent '{weather_agent.name}' is updated and ready!")

Agent 'weather_bot' is updated and ready!


deoploy agent

In [7]:
import os
import vertexai
from vertexai import agent_engines
from vertexai.agent_engines import AdkApp

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
location = os.getenv("GOOGLE_CLOUD_LOCATION")
staging_bucket = f"gs://{project_id}-agent-staging"

# 1. Initialize Vertex AI with your project and staging bucket
vertexai.init(
    project=project_id,
    location=location,
    staging_bucket=staging_bucket,
)

# 2. Wrap your ADK agent in an AdkApp
# We set enable_tracing=True so you can view its execution in Google Cloud Trace
agent_app = AdkApp(
    agent=weather_agent,
    enable_tracing=True
)

print("Deploying Weather Bot to Agent Platform... (This usually takes 3-5 minutes)")

# 3. Deploy to Agent Engine
remote_app = agent_engines.create(
    display_name="Weather Bot App",
    agent_engine=agent_app,  # FIX 1: Change 'agent' to 'agent_engine'
    requirements=[           # FIX 2: Pass the list directly to 'requirements'
        "google-cloud-aiplatform[agent_engines,adk]>=1.112",
        "requests",
        "geopy"
    ]
)

print(f"Deployment successful! Agent endpoint ID: {remote_app.resource_name}")

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '2.6.0', 'cloudpickle': '3.1.2', 'pydantic': '2.13.4'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.13.4'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.112', 'requests', 'geopy', 'cloudpickle==3.1.2', 'pydantic==2.13.4']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-04-23b9cb055c5f-agent-staging


Deploying Weather Bot to Agent Platform... (This usually takes 3-5 minutes)


INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-04-23b9cb055c5f-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-23b9cb055c5f-agent-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-23b9cb055c5f-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/410915331479/locations/us-central1/reasoningEngines/2878752613837307904/operations/5202186653749215232
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-04-23b9cb055c5f
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/410915331479/locations/us-central1/reasoningEngines/2878752613837307904
INFO:vertexai.agent_engines:To use this AgentEngine in another session:
IN

Deployment successful! Agent endpoint ID: projects/410915331479/locations/us-central1/reasoningEngines/2878752613837307904
